In [2]:
import pandas as pd

In [13]:
df = pd.read_csv("per.csv", sep="\t", header=None, engine="python")
df = df.drop(columns=[0])
df.columns = ["title", "category", "label"]
df.head()

,title,category,label
0,content,label,label_id
1,حسن جوهرچی بازیگر سینما و تلویزیون ایران در گف...,فرهنگی هنری,5
2,به گزارش گروه بین الملل باشگاه خبرنگاران جوان ...,بین الملل,2
3,به گزارش خبرنگار فوتبال و فوتسال گروه ورزشی با...,ورزشی,6
4,به‌ گزارش گروه اقتصادی باشگاه خبرنگاران به نقل...,اقتصادی,1


In [ ]:
with open('stopwords.txt', encoding='utf8') as stopwords_file:
    stopwords = stopwords_file.readlines()
stopwords = [line.replace('\n','') for line in stopwords]
stopwords

['﷼',
 '\u200f\u200f\u200fعلاقه مند',
 '\u200fیاب',
 '\u200fگیر',
 '\u200fگوی',
 '\u200fکن',
 '\u200fشو',
 '\u200fدیگران',
 '\u200fدار',
 '\u200fخواه',
 '\u200fتوان',
 '\u200fباش',
 '\u200fآی',
 '\u200fآور',
 'یکی',
 'یکهزار',
 'یکسال',
 'یکریز',
 'یکدیگر',
 'یک کمی',
 'یک کم',
 'یک چیزی',
 'یک جوری',
 'یک',
 'یواش یواش',
 'یواش',
 'یو',
 'یه',
 'یكی',
 'یكپارچه',
 'یكنواخت',
 'یكطرفه',
 'یكسری',
 'یكسره',
 'یكسال',
 'یكزمان',
 'یكریز',
 'یكدیگر',
 'یكدیر',
 'یكدم',
 'یكجوری',
 'یكجور',
 'یكجانبه',
 'یكجا',
 'یكباره',
 'یكبار',
 'یكایك',
 'یك',
 'یقیناً',
 'یقینا',
 'یعنی',
 'یشتری',
 'یشتر',
 'یش',
 'یست',
 'یری',
 'یرونِ',
 'یرد',
 'یافتیم',
 'یافتید',
 'یافتی',
 'یافته',
 'یافتن',
 'یافتم',
 'یافت',
 'یارب',
 'یاد',
 'یابیم',
 'یابید',
 'یابی',
 'یابند',
 'یابم',
 'یابد',
 'یاب',
 'یااینكه',
 'یاانكه',
 'یااز',
 'یا',
 'ی',
 'گیریم',
 'گیرید',
 'گیری',
 'گیرند',
 'گیرم',
 'گیرد',
 'گیر',
 'گيري',
 'گيرد',
 'گوییم',
 'گویید',
 'گویی',
 'گویند',
 'گویم',
 'گوید',
 'گویان',
 'گویا',
 '

In [5]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Amir\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [6]:
nltk_stopwords = nltk.corpus.stopwords.words('english')
stopwords.extend(nltk_stopwords)
len(stopwords)

2633

In [7]:
from parsivar import FindStems
stemmer = FindStems()


In [9]:
stemmer.convert_to_stem('کتاب ها')


'کتاب'

In [10]:
from parsivar import Tokenizer


In [ ]:
from parsivar import Tokenizer, FindStems
import pandas as pd

tokenizer = Tokenizer()
stemmer = FindStems()

dataset = pd.DataFrame(columns=('title_body', 'category'))

for index, row in df.iterrows():
    text = row['title'] 
    tokens = tokenizer.tokenize_words(text)
    tokens_filtered = [w for w in tokens if w not in stopwords]
    tokens_stemmed = [stemmer.convert_to_stem(w) for w in tokens_filtered]

    dataset.loc[index] = {
        'title_body': ' '.join(tokens_stemmed),
        'category': row['category']
    }



In [18]:
dataset.head()

,title_body,category
0,content,label
1,حسن جوهر بازیگر سینما تلویزیون ایران گفتگو خبر...,فرهنگی هنری
2,گزارش گروه الملل باشگاه خبرنگار جوان نقل هیل، ...,بین الملل
3,گزارش خبرنگار فوتبال فوتسال گروه ورزشی باشگاه ...,ورزشی
4,گزارش گروه اقتصادی باشگاه خبرنگار نقل پایگاه ا...,اقتصادی


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [20]:
vectorizer = TfidfVectorizer()
vectorizer.fit(dataset['title_body'])


TfidfVectorizer()

In [21]:
X = vectorizer.transform(dataset['title_body'])

In [22]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 208902 stored elements and shape (1645, 24102)>

In [23]:
from sklearn.preprocessing import LabelEncoder

In [24]:
le = LabelEncoder()
y = le.fit_transform(dataset['category'])

In [25]:
le.classes_

array(['label', 'اجتماعی', 'اقتصادی', 'بین الملل', 'سیاسی', 'علمی فناوری',
       'فرهنگی هنری', 'ورزشی', 'پزشکی'], dtype=object)

In [26]:
len(y)

1645

In [27]:
import numpy as np

In [28]:
print(np.unique(dataset['category']))

['label' 'اجتماعی' 'اقتصادی' 'بین الملل' 'سیاسی' 'علمی فناوری'
 'فرهنگی هنری' 'ورزشی' 'پزشکی']


In [30]:
np.shape(X)


(1645, 24102)

In [31]:
np.shape(y)

(1645,)

In [32]:
from sklearn.model_selection import train_test_split

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y)

In [35]:
from sklearn import svm

In [36]:
svmc = svm.SVC()
svmc.fit(X_train, y_train)

SVC()

In [37]:
svmc.score(X_test, y_test)

0.8009708737864077

In [39]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred = svmc.predict(X_test)

In [40]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.65      0.74      0.69        47
           2       0.81      0.76      0.78        38
           3       0.84      0.81      0.83        53
           4       0.69      0.62      0.65        50
           5       0.74      0.91      0.82        58
           6       0.90      0.90      0.90        58
           7       1.00      0.83      0.91        41
           8       0.85      0.79      0.82        67

    accuracy                           0.80       412
   macro avg       0.81      0.80      0.80       412
weighted avg       0.81      0.80      0.80       412

